# 16 · Diagnóstico de sobreajuste y subajuste — Grupo 17 · Semana 3
**Proyecto:** Sistema Inteligente de Detección y Clasificación de Phishing Híbrido (NLP + Análisis de URLs) · MIAR0545 · UEES
**Equipo:** Dennys Francisco Salazar Domínguez · Lenin Bolívar González Rivadeneira

**Entorno:** Colab con **GPU T4** (≈ 35 min en total). Reanudable: cada experimento guarda su historial y se omite si ya existe.
Requiere en Drive: `data/processed/emails_v1.parquet` (notebook 03) y `src/diagnostico.py`.

| Sección de la actividad | Dónde |
|---|---|
| Carga y exploración del dataset | Celdas 1–2 |
| Sistema de tracking (Keras-like callbacks / logging manual PyTorch / sklearn) | Celda 3 + `src/diagnostico.py` |
| Modelo base con logging: regresión logística (SGD) y DistilBERT | Celdas 4 y 7 |
| Curvas avanzadas: learning curve y validation curve (scikit-learn) | Celdas 5–6 |
| Diagnóstico cuantitativo | Celda 8 |
| Estrategias de mejora (≥ 2) y comparación antes/después | Celdas 9–10 |
| Conclusiones y RESUMEN | Celda 11 |

**Diseño del experimento.** Todos los modelos se entrenan en el fold `loso_phishing_pot` del proyecto: la fuente phishing_pot
queda COMPLETA fuera del entrenamiento. Así cada curva tiene tres conjuntos: entrenamiento, validación de la misma distribución
y una **fuente no vista** (recall de phishing sobre phishing_pot). El texto es `texto_modelo` (defangeado, URLs/IPs/correos como marcadores).

In [13]:
# Celda 1: Entorno, semillas y rutas
import os, sys, json, time, random, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17
def fijar(s=SEED):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError: pass
fijar()
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
OUT = BASE / "results" / "semana3"; FIG = OUT / "figuras"
for p in (OUT, FIG): p.mkdir(parents=True, exist_ok=True)
spec = importlib.util.spec_from_file_location("diagnostico", BASE / "src/diagnostico.py")
dg = importlib.util.module_from_spec(spec); sys.modules["diagnostico"] = dg; spec.loader.exec_module(dg)
FOLD = "loso_phishing_pot"
CLASES = ["legitimo", "spam", "phishing"]; PH = 2
print(ENV, "| salida:", OUT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
colab | salida: /content/drive/MyDrive/PI_G17/results/semana3


In [2]:
# Celda 2: Carga y exploración del dataset
D = pd.read_parquet(BASE / "data/processed/emails_v1.parquet",
                    columns=["id", "grupo", "clase", "y", "texto_modelo", f"fold_{FOLD}", f"fuera_{FOLD}"])
D["particion"] = np.where(D[f"fuera_{FOLD}"], "fuente_no_vista", D[f"fold_{FOLD}"])
TR = D[D.particion == "train"]; VA = D[D.particion == "val"]; OOD = D[D.particion == "fuente_no_vista"]
D["len_palabras"] = D["texto_modelo"].str.split().str.len()
EXPLORACION = {"n_total": int(len(D)),
               "clase_x_particion": pd.crosstab(D["particion"], D["clase"]).to_dict(),
               "origenes_en_train": sorted(TR["grupo"].unique().tolist()),
               "palabras_mediana_por_clase": D.groupby("clase")["len_palabras"].median().to_dict(),
               "palabras_p95": float(D["len_palabras"].quantile(.95))}
print(json.dumps(EXPLORACION, indent=1, ensure_ascii=False))

{
 "n_total": 54684,
 "clase_x_particion": {
  "legitimo": {
   "fuente_no_vista": 0,
   "test": 3207,
   "train": 14965,
   "val": 3207
  },
  "phishing": {
   "fuente_no_vista": 3702,
   "test": 932,
   "train": 4352,
   "val": 933
  },
  "spam": {
   "fuente_no_vista": 0,
   "test": 3508,
   "train": 16370,
   "val": 3508
  }
 },
 "origenes_en_train": [
  "CEAS_08",
  "Nigerian_Fraud",
  "SpamAssasin",
  "nazario_anual"
 ],
 "palabras_mediana_por_clase": {
  "legitimo": 186.0,
  "phishing": 203.0,
  "spam": 52.0
 },
 "palabras_p95": 726.0
}


In [3]:
# Celda 3: Sistema de tracking
# dg.RegistroMetricas cumple el papel de un callback de Keras dentro de loops manuales de PyTorch y de SGD:
# en cada evaluación registra loss y score de entrenamiento, validación y fuente no vista.
# Para scikit-learn se usan además learning_curve y validation_curve (Celdas 5 y 6).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss, f1_score, recall_score
def evaluar_sklearn(modelo, X, y):
    P = modelo.predict_proba(X)
    return log_loss(y, P, labels=[0, 1, 2]), f1_score(y, P.argmax(1), average="macro", labels=[0, 1, 2], zero_division=0)
def recall_ph(pred):   # fuente no vista: todo es phishing -> recall de phishing
    return float((np.asarray(pred) == PH).mean())
rng = np.random.RandomState(SEED)
idx_tr_eval = rng.choice(len(TR), min(3000, len(TR)), replace=False)   # train medido en modo evaluación (no running loss)
VEC = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=100_000, sublinear_tf=True).fit(TR["texto_modelo"])
Xtr, Xva, Xood = VEC.transform(TR["texto_modelo"]), VEC.transform(VA["texto_modelo"]), VEC.transform(OOD["texto_modelo"])
ytr, yva = TR["y"].values, VA["y"].values
print("TF-IDF:", Xtr.shape)

TF-IDF: (35687, 100000)


In [4]:
# Celda 4: Modelo base 1: regresión logística entrenada por SGD (logging por época)
def entrenar_sgd(alpha, epocas, nombre, early_stopping=False, paciencia=2):
    ruta = OUT / f"hist_{nombre}.json"
    if ruta.exists(): return dg.RegistroMetricas.cargar(ruta)
    fijar(); reg = dg.RegistroMetricas(nombre)
    m = SGDClassifier(loss="log_loss", alpha=alpha, learning_rate="optimal", random_state=SEED)
    mejor, sin, estado = np.inf, 0, None
    for e in range(1, epocas + 1):
        orden = np.random.RandomState(SEED + e).permutation(len(ytr))
        m.partial_fit(Xtr[orden], ytr[orden], classes=[0, 1, 2])
        tl, ts = evaluar_sklearn(m, Xtr[idx_tr_eval], ytr[idx_tr_eval]); vl, vs = evaluar_sklearn(m, Xva, yva)
        reg.log(epoca=e, train_loss=tl, val_loss=vl, train_score=ts, val_score=vs, ood_score=recall_ph(m.predict(Xood)))
        if early_stopping:
            if vl < mejor - 1e-4: mejor, sin, estado = vl, 0, (m.coef_.copy(), m.intercept_.copy(), e)
            else:
                sin += 1
                if sin >= paciencia: break
    if early_stopping and estado is not None: reg.log(epoca_restaurada=estado[2])
    reg.guardar(ruta); return reg
REG_SGD_BASE = entrenar_sgd(alpha=1e-7, epocas=25, nombre="sgd_base")   # regularización casi nula
dg.graficar_entrenamiento(REG_SGD_BASE.df().dropna(subset=["train_loss"]), "Regresión logística (SGD) · modelo base, α = 1e-7",
                          FIG / "01_sgd_base.png")
REG_SGD_BASE.df().tail(3)

,epoca,train_loss,val_loss,train_score,val_score,ood_score
22,23.0,0.001122,0.028080,1.0,0.991407,0.475419
23,24.0,0.001093,0.027713,1.0,0.991631,0.476769
24,25.0,0.001143,0.027858,1.0,0.991631,0.478390


In [5]:
# Celda 5: Curva avanzada C: learning curve por tamaño de dataset (scikit-learn)
from sklearn.model_selection import learning_curve, validation_curve, StratifiedKFold
from sklearn.pipeline import make_pipeline
ruta_lc = OUT / "learning_curve.json"
SUB = TR.sample(min(20000, len(TR)), random_state=SEED)    # muestra estratificable para acotar el tiempo
CV = StratifiedKFold(3, shuffle=True, random_state=SEED)
def pipe(alpha):
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=100_000, sublinear_tf=True),
                         SGDClassifier(loss="log_loss", alpha=alpha, max_iter=25, tol=None, random_state=SEED))
if ruta_lc.exists():
    LC = json.loads(ruta_lc.read_text())
else:
    t0 = time.time()
    tam, tr_s, va_s = learning_curve(pipe(1e-7), SUB["texto_modelo"], SUB["y"], cv=CV, scoring="f1_macro",
                                     train_sizes=np.linspace(0.1, 1.0, 5), n_jobs=-1, random_state=SEED, shuffle=True)
    LC = {"tamanos": tam.tolist(), "train": tr_s.tolist(), "val": va_s.tolist(), "segundos": round(time.time() - t0)}
    ruta_lc.write_text(json.dumps(LC))
dg.graficar_learning_curve(LC["tamanos"], np.array(LC["train"]), np.array(LC["val"]),
                           "C. Learning curve · regresión logística (SGD, α = 1e-7)", FIG / "03_learning_curve.png")
{"tamanos": LC["tamanos"], "val_media": np.round(np.mean(LC["val"], 1), 4).tolist()}

{'tamanos': [1333, 4333, 7333, 10333, 13333],
 'val_media': [0.9619, 0.978, 0.9821, 0.9837, 0.9861]}

In [6]:
# Celda 6: Curva avanzada D: validation curve por hiperparámetro (α de regularización L2)
ruta_vc = OUT / "validation_curve.json"
ALPHAS = [1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]
if ruta_vc.exists():
    VC = json.loads(ruta_vc.read_text())
else:
    t0 = time.time()
    tr_s, va_s = validation_curve(pipe(1e-7), SUB["texto_modelo"], SUB["y"], param_name="sgdclassifier__alpha",
                                  param_range=ALPHAS, cv=CV, scoring="f1_macro", n_jobs=-1)
    VC = {"alphas": ALPHAS, "train": tr_s.tolist(), "val": va_s.tolist(), "segundos": round(time.time() - t0)}
    ruta_vc.write_text(json.dumps(VC))
ALPHA_OPT = VC["alphas"][int(np.argmax(np.mean(VC["val"], 1)))]
dg.graficar_validation_curve(VC["alphas"], np.array(VC["train"]), np.array(VC["val"]),
                             "D. Validation curve · regularización L2 (α)", FIG / "04_validation_curve.png", param="α")
print("α óptimo por validación cruzada:", ALPHA_OPT)

α óptimo por validación cruzada: 1e-07


In [7]:
# Celda 7: Modelo base 2: DistilBERT con logging manual (PyTorch)
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEV.type == "cuda" or ENV == "local", "Activa GPU (Entorno de ejecución → T4)"
CFG = {"modelo": "distilbert-base-uncased", "max_len": 256, "batch": 32, "n_train": 20000, "evals_por_epoca": 2,
       "n_eval": 2000, "marcadores": ["__url__", "__ip__", "__email__", "__anio__", "__dominio_propio__"]}
def crear_tok():
    t = AutoTokenizer.from_pretrained(CFG["modelo"]); t.add_tokens(CFG["marcadores"]); return t
def crear_modelo(tok, dropout=0.1):
    m = AutoModelForSequenceClassification.from_pretrained(CFG["modelo"], num_labels=3, dropout=dropout, seq_classif_dropout=max(dropout, 0.2))
    m.resize_token_embeddings(len(tok)); return m

In [8]:
# Celda 7b: Loop de entrenamiento con tracking (base y regularizado)
TOK = crear_tok(); COL = DataCollatorWithPadding(TOK)
TR_BERT = TR.sample(min(CFG["n_train"], len(TR)), random_state=SEED)       # muestra declarada para acotar GPU
EV = {"train": TR_BERT.sample(min(CFG["n_eval"], len(TR_BERT)), random_state=SEED),
      "val": VA.sample(min(CFG["n_eval"], len(VA)), random_state=SEED),
      "ood": OOD.sample(min(1500, len(OOD)), random_state=SEED)}
def cod(df):
    e = TOK(df["texto_modelo"].tolist(), truncation=True, max_length=CFG["max_len"])
    return [{"input_ids": a, "attention_mask": m, "labels": int(y)} for a, m, y in zip(e["input_ids"], e["attention_mask"], df["y"])]
CACHE = {k: cod(v) for k, v in EV.items()}
@torch.no_grad()
def medir(m, k, smoothing=0.0):
    m.eval(); P, Y = [], []
    for b in DataLoader(CACHE[k], batch_size=128, collate_fn=COL):
        Y.append(b.pop("labels").numpy()); b = {kk: v.to(DEV) for kk, v in b.items()}
        with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
            P.append(torch.softmax(m(**b).logits.float(), -1).cpu().numpy())
    P, Y = np.concatenate(P), np.concatenate(Y); m.train()
    return log_loss(Y, P, labels=[0, 1, 2]), f1_score(Y, P.argmax(1), average="macro", labels=[0, 1, 2], zero_division=0), P.argmax(1)
def entrenar_bert(nombre, epocas, lr, weight_decay=0.0, dropout=0.1, label_smoothing=0.0, early_stopping=False, paciencia=2):
    ruta = OUT / f"hist_{nombre}.json"
    if ruta.exists(): return dg.RegistroMetricas.cargar(ruta)
    fijar(); reg = dg.RegistroMetricas(nombre); m = crear_modelo(TOK, dropout).to(DEV)
    y = TR_BERT["y"].values; w = torch.tensor(len(y) / (3 * np.bincount(y, minlength=3)), dtype=torch.float, device=DEV)
    dl = DataLoader(cod(TR_BERT), batch_size=CFG["batch"], shuffle=True, collate_fn=COL, generator=torch.Generator().manual_seed(SEED))
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=weight_decay)
    tot = len(dl) * epocas; sch = get_linear_schedule_with_warmup(opt, int(0.1 * tot), tot)
    sc = torch.amp.GradScaler("cuda", enabled=DEV.type == "cuda")
    lossf = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=label_smoothing)
    cada = max(1, len(dl) // CFG["evals_por_epoca"]); paso, mejor, sin, mejor_estado, t0 = 0, np.inf, 0, None, time.time()
    parar = False
    for e in range(epocas):
        for b in dl:
            b = {k: v.to(DEV) for k, v in b.items()}; yy = b.pop("labels")
            with torch.autocast(device_type=DEV.type, dtype=torch.float16, enabled=DEV.type == "cuda"):
                loss = lossf(m(**b).logits.float(), yy)
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); sc.step(opt); sc.update(); sch.step(); paso += 1
            if paso % cada == 0:
                tl, ts, _ = medir(m, "train"); vl, vs, _ = medir(m, "val"); _, _, po = medir(m, "ood")
                reg.log(epoca=round(paso / len(dl), 2), paso=paso, train_loss=tl, val_loss=vl, train_score=ts, val_score=vs,
                        ood_score=recall_ph(po), running_loss=float(loss), segundos=round(time.time() - t0))
                print(nombre, reg.filas[-1])
                if early_stopping:
                    if vl < mejor - 1e-4: mejor, sin, mejor_estado = vl, 0, (paso, {k: v.detach().cpu().clone() for k, v in m.state_dict().items()})
                    else:
                        sin += 1
                        if sin >= paciencia: parar = True; break
        if parar: break
    if early_stopping and mejor_estado is not None:
        reg.log(paso_restaurado=mejor_estado[0])
    reg.guardar(ruta); del m; torch.cuda.empty_cache() if DEV.type == "cuda" else None
    return reg
REG_BERT_BASE = entrenar_bert("bert_base", epocas=5, lr=5e-5)      # sin regularización explícita ni early stopping
dg.graficar_entrenamiento(REG_BERT_BASE.df().dropna(subset=["train_loss"]), "DistilBERT · modelo base (5 épocas, sin regularización)",
                          FIG / "02_bert_base.png")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion

bert_base {'epoca': 0.5, 'paso': 312.0, 'train_loss': 0.05815313234721462, 'val_loss': 0.060773341476824436, 'train_score': 0.9796528318024827, 'val_score': 0.9773369683139764, 'ood_score': 0.356, 'running_loss': 0.007641658652573824, 'segundos': 61.0}
bert_base {'epoca': 1.0, 'paso': 624.0, 'train_loss': 0.03374084098957692, 'val_loss': 0.041918112667962616, 'train_score': 0.9896084500792903, 'val_score': 0.9842140651858426, 'ood_score': 0.468, 'running_loss': 0.02170012705028057, 'segundos': 127.0}
bert_base {'epoca': 1.5, 'paso': 936.0, 'train_loss': 0.025308465982568012, 'val_loss': 0.025400840783888618, 'train_score': 0.9904378201671605, 'val_score': 0.9918063978223812, 'ood_score': 0.43066666666666664, 'running_loss': 0.0045116194523870945, 'segundos': 196.0}
bert_base {'epoca': 2.0, 'paso': 1248.0, 'train_loss': 0.018204255282039963, 'val_loss': 0.028312774425251937, 'train_score': 0.9943161894502909, 'val_score': 0.991820098107325, 'ood_score': 0.504, 'running_loss': 0.18960128

In [9]:
# Celda 8: Diagnóstico cuantitativo de los modelos base
limpio = lambda r: r.df().dropna(subset=["train_loss"])
DIAG_BASE = {"sgd_base": dg.diagnosticar(limpio(REG_SGD_BASE)), "bert_base": dg.diagnosticar(limpio(REG_BERT_BASE))}
lc_gap = float(np.mean(LC["train"], 1)[-1] - np.mean(LC["val"], 1)[-1])
lc_pend = float(np.polyfit(LC["tamanos"][-3:], np.mean(LC["val"], 1)[-3:], 1)[0] * 1000)
DIAG_BASE["learning_curve"] = {"gap_final": round(lc_gap, 4), "pendiente_val_por_1000_correos_ultimos_3": round(lc_pend, 5)}
DIAG_BASE["validation_curve"] = {"alpha_opt": ALPHA_OPT, "val_por_alpha": dict(zip(map(str, VC["alphas"]), np.round(np.mean(VC["val"], 1), 4).tolist())),
                                 "train_por_alpha": dict(zip(map(str, VC["alphas"]), np.round(np.mean(VC["train"], 1), 4).tolist()))}
print(json.dumps(DIAG_BASE, indent=1, ensure_ascii=False))

{
 "sgd_base": {
  "evaluaciones": 25,
  "train_loss_final": 0.00114,
  "val_loss_final": 0.02786,
  "train_score_final": 1.0,
  "val_score_final": 0.9916,
  "gap_loss_final": 0.02672,
  "gap_score_final": 0.0084,
  "pendiente_gap_loss_2a_mitad": -0.000287,
  "val_loss_min": 0.02734,
  "epoca_val_loss_min": 22.0,
  "subida_val_loss_sobre_min_rel": 0.019,
  "train_loss_baja_tras_min": true,
  "ood_score_final": 0.4784,
  "ood_score_max": 0.5837,
  "epoca_ood_max": 1.0,
  "gap_val_ood_final": 0.5132,
  "caida_ood_desde_max": 0.1053,
  "criterios": {
   "val_loss_sube_y_train_baja": false,
   "gap_score_alto_y_creciente": false,
   "train_score_bajo": false
  },
  "diagnostico": "optimo_en_distribucion_overfitting_a_fuente"
 },
 "bert_base": {
  "evaluaciones": 10,
  "train_loss_final": 0.00195,
  "val_loss_final": 0.02971,
  "train_score_final": 0.9982,
  "val_score_final": 0.9929,
  "gap_loss_final": 0.02777,
  "gap_score_final": 0.0053,
  "pendiente_gap_loss_2a_mitad": 0.003574,
  "val

In [10]:
# Celda 9: Estrategias de mejora implementadas
# E1 (DistilBERT, overfitting): early stopping sobre val loss (paciencia 2 evaluaciones) + regularización:
#    weight decay 0,05 (L2 desacoplado), dropout 0,2 y label smoothing 0,1; lr reducido a 3e-5.
# E2 (regresión logística, overfitting): regularización L2 elegida por la validation curve (α óptimo) + early stopping.
REG_BERT_MEJ = entrenar_bert("bert_regularizado", epocas=5, lr=3e-5, weight_decay=0.05, dropout=0.2,
                             label_smoothing=0.1, early_stopping=True, paciencia=2)
REG_SGD_MEJ = entrenar_sgd(alpha=ALPHA_OPT, epocas=25, nombre="sgd_regularizado", early_stopping=True, paciencia=2)
dg.graficar_entrenamiento(limpio(REG_BERT_MEJ), "DistilBERT · con early stopping + regularización", FIG / "05_bert_regularizado.png")
dg.graficar_entrenamiento(limpio(REG_SGD_MEJ), f"Regresión logística (SGD) · α = {ALPHA_OPT:g} + early stopping", FIG / "06_sgd_regularizado.png")
dg.graficar_comparacion(limpio(REG_BERT_BASE), limpio(REG_BERT_MEJ), "Antes / después · DistilBERT (val loss)", FIG / "07_comparacion_bert.png")
dg.graficar_comparacion(limpio(REG_SGD_BASE), limpio(REG_SGD_MEJ), "Antes / después · regresión logística (val loss)", FIG / "08_comparacion_sgd.png")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert_regularizado {'epoca': 0.5, 'paso': 312.0, 'train_loss': 0.18176512057723768, 'val_loss': 0.1834156778136026, 'train_score': 0.9785630319132116, 'val_score': 0.9818370732028434, 'ood_score': 0.42, 'running_loss': 0.42943504452705383, 'segundos': 70.0}
bert_regularizado {'epoca': 1.0, 'paso': 624.0, 'train_loss': 0.17108390878126073, 'val_loss': 0.17980529416347707, 'train_score': 0.9886271962261443, 'val_score': 0.9805085302177999, 'ood_score': 0.44466666666666665, 'running_loss': 0.38263821601867676, 'segundos': 141.0}
bert_regularizado {'epoca': 1.5, 'paso': 936.0, 'train_loss': 0.1472841481953509, 'val_loss': 0.15590145315059062, 'train_score': 0.9895555481056961, 'val_score': 0.9864718545637432, 'ood_score': 0.57, 'running_loss': 0.5249348282814026, 'segundos': 212.0}
bert_regularizado {'epoca': 2.0, 'paso': 1248.0, 'train_loss': 0.13357900709090845, 'val_loss': 0.14225553019339185, 'train_score': 0.9948248063156618, 'val_score': 0.99014939243378, 'ood_score': 0.46533333333333

In [11]:
# Celda 10: Comparación antes / después
DIAG_MEJ = {"sgd_regularizado": dg.diagnosticar(limpio(REG_SGD_MEJ)), "bert_regularizado": dg.diagnosticar(limpio(REG_BERT_MEJ))}
def fila(nombre, d):
    return {"modelo": nombre, "diagnostico": d["diagnostico"], "val_loss_min": d["val_loss_min"], "val_loss_final": d["val_loss_final"],
            "subida_val_loss_%": round(100 * d["subida_val_loss_sobre_min_rel"], 1), "gap_loss_final": d["gap_loss_final"],
            "gap_score_final": d["gap_score_final"], "val_f1_final": d["val_score_final"], "fuente_no_vista_recall_final": d.get("ood_score_final")}
TABLA = pd.DataFrame([fila("sgd_base", DIAG_BASE["sgd_base"]), fila("sgd_regularizado", DIAG_MEJ["sgd_regularizado"]),
                      fila("bert_base", DIAG_BASE["bert_base"]), fila("bert_regularizado", DIAG_MEJ["bert_regularizado"])])
TABLA.to_csv(OUT / "comparacion_antes_despues.csv", index=False)
TABLA

,modelo,diagnostico,val_loss_min,val_loss_final,subida_val_loss_%,gap_loss_final,gap_score_final,val_f1_final,fuente_no_vista_recall_final
0,sgd_base,optimo_en_distribucion_overfitting_a_fuente,0.02734,0.02786,1.9,0.02672,0.0084,0.9916,0.4784
1,sgd_regularizado,optimo_en_distribucion_overfitting_a_fuente,0.04754,0.04762,0.2,0.04519,0.0077,0.9917,0.5348
2,bert_base,overfitting,0.02263,0.02971,31.3,0.02777,0.0053,0.9929,0.5087
3,bert_regularizado,optimo_en_distribucion_overfitting_a_fuente,0.13601,0.13837,1.7,0.00699,0.0033,0.9919,0.6167


In [12]:
# Celda 11: Conclusiones y RESUMEN
# Las conclusiones se derivan de estas cifras (sección 6 del reporte); este bloque las deja trazables.
RESUMEN = {"entorno": ENV, "fold": FOLD, "exploracion": EXPLORACION, "cfg_bert": {k: v for k, v in CFG.items() if k != "marcadores"},
           "diagnostico_base": DIAG_BASE, "diagnostico_mejorado": DIAG_MEJ, "comparacion": TABLA.to_dict(orient="records"),
           "historial_bert_base": limpio(REG_BERT_BASE)[["epoca", "train_loss", "val_loss", "train_score", "val_score", "ood_score"]].round(4).to_dict(orient="list"),
           "historial_bert_regularizado": limpio(REG_BERT_MEJ)[["epoca", "train_loss", "val_loss", "train_score", "val_score", "ood_score"]].round(4).to_dict(orient="list"),
           "figuras": sorted(p.name for p in FIG.glob("*.png"))}
(OUT / "resumen_semana3.json").write_text(json.dumps(RESUMEN, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (S3) =====")
print(json.dumps(RESUMEN, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (S3) =====
{"entorno":"colab","fold":"loso_phishing_pot","exploracion":{"n_total":54684,"clase_x_particion":{"legitimo":{"fuente_no_vista":0,"test":3207,"train":14965,"val":3207},"phishing":{"fuente_no_vista":3702,"test":932,"train":4352,"val":933},"spam":{"fuente_no_vista":0,"test":3508,"train":16370,"val":3508}},"origenes_en_train":["CEAS_08","Nigerian_Fraud","SpamAssasin","nazario_anual"],"palabras_mediana_por_clase":{"legitimo":186.0,"phishing":203.0,"spam":52.0},"palabras_p95":726.0},"cfg_bert":{"modelo":"distilbert-base-uncased","max_len":256,"batch":32,"n_train":20000,"evals_por_epoca":2,"n_eval":2000},"diagnostico_base":{"sgd_base":{"evaluaciones":25,"train_loss_final":0.00114,"val_loss_final":0.02786,"train_score_final":1.0,"val_score_final":0.9916,"gap_loss_final":0.02672,"gap_score_final":0.0084,"pendiente_gap_loss_2a_mitad":-0.000287,"val_loss_min":0.02734,"epoca_val_loss_min":22.0,"subida_val_loss_sobre_min_rel":0.019,"train_loss_baja_tras_min":true,"ood_scor